# Guided EDA — competition AMEX mart (slice 1)

**Collaborative session cut** — load → drop high-null columns → numeric vs categorical → class imbalance.

Out of this notebook: WOE/IV, modeling, correlations beyond the balance check.

**Disclaimer:** public demo data only (not live bank BFSI).

> **Warning — book-sample mart ≠ competition prepare contract.**  
> `data/credit/application_mart/application_mart.parquet` comes from `build-application-mart` on the book pickle. Most AMEX category codes stay as floats there, so a dtype-only inventory under-counts categoricals (often only `D_63_last` / `D_64_last`).  
> This guided session loads the **competition AMEX mart** (`build-competition-mart --source-kind amex`) where official categoricals are aggregated as `{col}_mode` / `{col}_last` per [#157](https://github.com/JLaborda/SmartWealthAI/issues/157).

| Column | Meaning |
| --- | --- |
| `customer_ID` | Application / customer id (not a feature) |
| `target` | `1` = bad (default), `0` = good |

## 1. Load data

Resolution order for **this guided session**:

1. `data/credit/application_mart/amex/application_mart.parquet` (**preferred** — competition AMEX mart, `#157` prepare contract)
2. Else `data/credit/application_mart/application_mart.parquet` (book-sample mart — **warn**: wrong categorical contract)
3. Else `credit/data/train_df_sample.pkl` (Git LFS book sample)

Rebuild competition mart when raw is present:

```bash
poetry run credit-css build-competition-mart \
  --source-kind amex \
  --raw-dir data/credit/raw/amex \
  --output-dir data/credit/application_mart/amex
```

In [ ]:
from pathlib import Path

import pandas as pd

ID_COL = "customer_ID"
TARGET = "target"
NULL_DROP_THRESHOLD = 0.80

# Official AMEX categoricals (competition docs / #157) → mart names {col}_mode / {col}_last
AMEX_CATEGORICALS = [
    "B_30",
    "B_38",
    "D_114",
    "D_116",
    "D_117",
    "D_120",
    "D_126",
    "D_63",
    "D_64",
    "D_66",
    "D_68",
]
AMEX_CAT_FEATURE_NAMES = {
    f"{c}_{suf}" for c in AMEX_CATEGORICALS for suf in ("mode", "last")
}

REPO_ROOT_CANDIDATES = [Path("../.."), Path(".")]


def resolve_repo_root() -> Path:
    for root in REPO_ROOT_CANDIDATES:
        if (root / "credit" / "notebooks").exists() or (root / "pyproject.toml").exists():
            return root.resolve()
    return Path("../..").resolve()


ROOT = resolve_repo_root()
COMPETITION_AMEX_MART = (
    ROOT / "data" / "credit" / "application_mart" / "amex" / "application_mart.parquet"
)
BOOK_MART = ROOT / "data" / "credit" / "application_mart" / "application_mart.parquet"
SAMPLE_PKL = ROOT / "credit" / "data" / "train_df_sample.pkl"

if COMPETITION_AMEX_MART.exists():
    source_path = COMPETITION_AMEX_MART
    source_label = "competition AMEX mart (preferred for this guided session)"
    df = pd.read_parquet(source_path)
elif BOOK_MART.exists():
    source_path = BOOK_MART
    source_label = (
        "book-sample application mart (FALLBACK — wrong categorical contract; "
        "rebuild competition mart)"
    )
    df = pd.read_parquet(source_path)
elif SAMPLE_PKL.exists() and SAMPLE_PKL.stat().st_size > 1_000_000:
    source_path = SAMPLE_PKL
    source_label = "book AMEX sample pickle (fallback — not the competition mart)"
    df = pd.read_pickle(source_path)
else:
    raise FileNotFoundError(
        "No usable mart/sample found.\n"
        f"  competition: {COMPETITION_AMEX_MART}\n"
        f"  book mart:   {BOOK_MART}\n"
        f"  pkl:         {SAMPLE_PKL}\n"
        "Run build-competition-mart or git lfs pull --include=\"credit/data/*\""
    )

print(f"source: {source_label}")
print(f"path:   {source_path}")
print(f"shape:  {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"id col present: {ID_COL in df.columns}; target present: {TARGET in df.columns}")
df.head(3)

## 2. Drop columns with ≥80% nulls

High missingness columns add little signal for a first pass and inflate the feature list.
We compute the null rate per column, drop those at or above **80%**, and keep a cleaned frame `df_clean`.

Id and target are never dropped here (they are required keys).

In [ ]:
rows_before, cols_before = df.shape
null_rate = df.isna().mean().sort_values(ascending=False)

print("Null-rate summary (all columns):")
print(null_rate.describe().to_string())
print()
print("Top 15 null rates:")
print(null_rate.head(15).to_string())

drop_cols = [
    c
    for c in null_rate.index[null_rate >= NULL_DROP_THRESHOLD]
    if c not in {ID_COL, TARGET}
]
df_clean = df.drop(columns=drop_cols)
rows_after, cols_after = df_clean.shape

print()
print(f"Dropped {len(drop_cols)} columns with null rate ≥ {NULL_DROP_THRESHOLD:.0%}")
print("Dropped columns:")
for c in drop_cols:
    print(f"  - {c}: {null_rate[c]:.1%}")
print()
print(f"Shape before: {rows_before:,} × {cols_before:,}")
print(f"Shape after:  {rows_after:,} × {cols_after:,}")

## 3. Numeric vs categorical features

**Rule for this guided session (competition AMEX contract + dtype):**

- Exclude `customer_ID` and `target` from feature lists.
- **Categorical features** = union of:
  1. Official AMEX prepare-contract aggregates present in the frame: `{col}_mode` / `{col}_last` for  
     `B_30`, `B_38`, `D_114`, `D_116`, `D_117`, `D_120`, `D_126`, `D_63`, `D_64`, `D_66`, `D_68`  
     (even when stored as float codes — dtype alone is misleading on the book-sample mart).
  2. Any other remaining non-numeric dtypes (`object` / `string` / `category`).
- **Numeric features:** everything else among the feature columns.

We also print a **dtype-only** count for comparison (book-sample path often shows only ~2).

In [ ]:
def classify_features(frame: pd.DataFrame):
    cols = [c for c in frame.columns if c not in {ID_COL, TARGET}]
    dtype_num = set(frame[cols].select_dtypes(include="number").columns)
    dtype_cat = [c for c in cols if c not in dtype_num]
    contract_cat = sorted(c for c in cols if c in AMEX_CAT_FEATURE_NAMES)
    cats = sorted(set(dtype_cat) | set(contract_cat))
    nums = [c for c in cols if c not in cats]
    return cols, cats, nums, dtype_cat, contract_cat


# Inventory on raw loaded frame (before null drop) vs cleaned frame
_, cats_before, _, dtype_before, contract_before = classify_features(df)
feature_cols, categorical_features, numeric_features, dtype_categorical, contract_categorical = (
    classify_features(df_clean)
)
cats_dropped_by_null = [c for c in cats_before if c in drop_cols]

print("=== BEFORE ≥80% null drop ===")
print(f"Categorical (contract ∪ non-numeric): {len(cats_before)}")
print("List:", ", ".join(cats_before))
print(f"dtype-only non-numeric: {len(dtype_before)} → {dtype_before}")
print()
print(f"Categoricals dropped by null threshold: {len(cats_dropped_by_null)} → {cats_dropped_by_null}")
print()
print("=== AFTER ≥80% null drop ===")
print(f"Feature columns (excl. id/target): {len(feature_cols)}")
print(f"Categorical features (contract ∪ non-numeric dtype): {len(categorical_features)}")
print(f"  of which official AMEX *_mode/*_last present: {len(contract_categorical)}")
print(f"  of which dtype-only non-numeric: {len(dtype_categorical)}")
print(f"Numeric features (remainder): {len(numeric_features)}")
print()
print("Categorical list:")
for c in categorical_features:
    print(f"  - {c}  dtype={df_clean[c].dtype}")
print()
print("Numeric sample:", numeric_features[:12])
print()
print("Dtype value counts (feature columns only):")
print(df_clean[feature_cols].dtypes.value_counts().to_string())

## 4. Class imbalance (`target`)

`target = 1` → **bad** (default). `target = 0` → **good**.

Credit default datasets are usually **imbalanced** (far more goods than bads). We report counts, rates, and a simple bar chart so the imbalance is visible before any modeling.

In [ ]:
BAD, GOOD = 1, 0

counts = df_clean[TARGET].value_counts(dropna=False).sort_index()
n_good = int((df_clean[TARGET] == GOOD).sum())
n_bad = int((df_clean[TARGET] == BAD).sum())
n_total = n_good + n_bad
pct_good = n_good / n_total
pct_bad = n_bad / n_total

balance = pd.DataFrame(
    {
        "label": ["good (0)", "bad (1)"],
        "n": [n_good, n_bad],
        "pct": [pct_good, pct_bad],
    }
)
print(balance.to_string(index=False, formatters={"pct": "{:.2%}".format}))
print()
print(f"Imbalance ratio (good:bad) ≈ {n_good / max(n_bad, 1):.2f}:1")
print(f"Default / bad rate: {pct_bad:.2%}")

ax = counts.plot(kind="bar", title="target balance (0=good, 1=bad)", rot=0, figsize=(5, 3))
ax.set_xlabel("target")
ax.set_ylabel("count")

## Slice 1 summary (verified local run — competition AMEX mart)

- **Source:** competition AMEX mart — `data/credit/application_mart/amex/application_mart.parquet` (rebuilt with `build-competition-mart`; #157 contract)
- **Shape before null drop:** 458,913 rows × 909 columns
- **Columns dropped (≥80% null):** 107 (includes categorical `D_66_mode`, `D_66_last`)
- **Shape after null drop:** 458,913 rows × 802 columns
- **Categoricals before null drop:** 22 (all official `{col}_mode`/`{col}_last`)
- **Categoricals after null drop:** 20 (lost `D_66_mode`, `D_66_last`); numeric features: 780
- **dtype-only non-numeric:** only `D_63_*` / `D_64_*` (strings); other official cats are float-coded `*_mode`/`*_last` — hence contract ∪ dtype rule
- **Class balance:** good (0) ≈ 340,085 (74.11%); bad (1) ≈ 118,828 (25.89%)

**Stop here for this session.** Next: deeper EDA (#153), WOE/IV / scoring (#145).